# D1 · Reading a PCA: loadings, explained variance and relationship with the diagnosis

**Type**: Interpret · **Level**: ●●○ independent · **Time**: about 45 minutes · **core path ★**

## Objective

Interpret a PCA on 30 measurements of breast nodules: how much variance each component captures, which variables define it (loadings), why the first component is almost a "general factor", how the explained variance links to the redundancy between variables
(three nearly identical size measurements) and whether the first component separates benign from malignant tumors.

## Dataset

`load_breast_cancer` (569 nodules, 30 features: mean, standard error and worst value of 10 measurements, target 1 = benign), standardized: `Z` (569 × 30), with `names` (the column names) and `y`.

## Educational purpose

Read the output of a PCA the way you would read a table in a paper: not "how many components" but **what** they represent, which columns move together and what it means to have 44% of the variance on the first component.

## Task

1. Train `PCA().fit(Z)` and save `ratio` (`explained_variance_ratio_`) and `k_for_threshold`: dictionary `{0.8: k, 0.9: k, 0.95: k}` with the minimum number of components to reach each cumulative variance threshold.
2. For the **first component**: `top_pc1` (list of the names of the 5 variables with the largest |loading|) and `pc1_sign_share` (fraction of loadings with the same sign as the sum of the loadings: if it is close to 1 all the variables contribute in the same direction).
3. **Redundancy:** `size_corr` (the minimum correlation between `mean radius`, `mean perimeter` and `mean area`) and `three_sizes_ratio` (variance explained by the first component of a PCA done only on those three standardized columns).
4. **Relationship with the diagnosis:** `diagnosis_corr` (array of 6 values: |correlation| of each of the first 6 components with `y`) and `most_correlated_pc` (order number, from 1). Then `acc_k` (dictionary `k → cross-validated accuracy` of `LogisticRegression` on the first `k` components, with `k` in `[1, 2, 5, 10, 30]`, `StratifiedKFold(5, shuffle=True, random_state=0)`).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer()
names, y = list(data.feature_names), data.target
Z = StandardScaler().fit_transform(data.data)
cv = StratifiedKFold(5, shuffle=True, random_state=0)

## Your work

In [ ]:
pca = PCA().fit(Z)
ratio = None
k_for_threshold = None

In [ ]:
top_pc1 = None
pc1_sign_share = None

In [ ]:
size_corr = None
three_sizes_ratio = None

In [ ]:
diagnosis_corr = None
most_correlated_pc = None
acc_k = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='Loadings: `pd.Series(pca.components_[0], index=names).abs().nlargest(5).index`. Sign: `np.sign(l.sum())` against `np.sign(l)`. Three sizes: `columns = [names.index(c) for c in ["mean radius", "mean perimeter", "mean area"]]`; `np.corrcoef(Z[:, columns].T).min()`; `PCA().fit(Z[:, columns]).explained_variance_ratio_[0]`. Components in the pipeline: `make_pipeline(PCA(k), LogisticRegression(max_iter=2000))` (the data are already standardized).'):
    check(close(ratio, pca.explained_variance_ratio_) and 0.42 < ratio[0] < 0.46 and 0.18 < ratio[1] < 0.2, "the first component explains 44% of the variance, the second 19%", "explained_variance_ratio_")
    check(k_for_threshold == {0.8: 5, 0.9: 7, 0.95: 10}, "5 components for 80%, 7 for 90%, 10 for 95% (out of 30 variables)", "np.searchsorted(np.cumsum(ratio), threshold) + 1")
    check(len(top_pc1) == 5 and "mean concave points" in top_pc1 and pc1_sign_share > 0.85, "the variables with the largest weight describe shape and concavity; almost all variables contribute with the same sign", "np.abs(pca.components_[0]) sorted, and the sign of each loading")
    check(size_corr > 0.98 and three_sizes_ratio > 0.99, "radius, perimeter and area are almost the same variable: a single component explains over 99%", "minimum correlation between the three columns and PCA on those three")
    check(len(diagnosis_corr) == 6 and most_correlated_pc == 1 and diagnosis_corr[0] > 0.75 and (diagnosis_corr[1:] < 0.25).all(), "only the first component is linked to the diagnosis (correlation over 0.75); the others below 0.25", "np.corrcoef(scores[:, i], y)")
    check(set(acc_k) == {1, 2, 5, 10, 30} and acc_k[1] > 0.85 and acc_k[10] > acc_k[1] and acc_k[30] > 0.95, "a single component already gives over 0.85; with 10 components it improves; with 30 over 0.95", "LogisticRegression on PCA(k) in a pipeline")
done()

---
The commented solution is in `solutions/D_interpret/D1_reading_loadings_and_variance_on_tumors.ipynb`. Open it after you pass the check, or after you have really tried.